In [ ]:
import duckdb
import time

def extract_overture_india():
    print("Initializing DuckDB spatial engine...")
    con = duckdb.connect("overture_spatial.db")
    con.execute("INSTALL spatial; LOAD spatial;")
    con.execute("INSTALL httpfs; LOAD httpfs;")
    
    # Base S3 Path for Overture (August 2026 Stable Release)
    S3_BASE = "s3://overturemaps-us-west-2/release/2026-08-19.0"
    
    # Bounding Box for India
    INDIA_BBOX = "bbox.xmin >= 68.1 AND bbox.xmax <= 97.4 AND bbox.ymin >= 6.5 AND bbox.ymax <= 37.1"

    # --- 2. TRANSPORTATION (Roads & Connectors) ---
    print("Streaming Road Networks & Intersections...")
    # Segments (Lines)
    con.execute(f"""
        COPY (
            SELECT 
                id, 
                class, 
                road_surface AS surface,
                geometry
            FROM read_parquet('{S3_BASE}/theme=transportation/type=segment/*', filename=true, hive_partitioning=1)
            WHERE {INDIA_BBOX}
        ) TO 'india_roads.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
    """)
    
    # Connectors (Junction Nodes)
    con.execute(f"""
        COPY (
            SELECT id, geometry
            FROM read_parquet('{S3_BASE}/theme=transportation/type=connector/*', filename=true, hive_partitioning=1)
            WHERE {INDIA_BBOX}
        ) TO 'india_connectors.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
    """)

extract_overture_india()